# NumericVerifier V6.1 -- Final Demo Notebook

**System:** P&L-Aware Post-Generation Verification System for Numeric Reliability in LLM Outputs

**Model:** V6.1 XGBClassifier (10 features, 364 training cases, 98.18% test accuracy)

**Date:** 2026-03-29

---

## Project Scope

NumericVerifier is a post-generation verification system designed to detect numeric errors in LLM-generated answers about financial statements. Given a question, a P&L table as evidence, and a candidate answer, the system:

1. **Extracts** numeric claims from the candidate answer
2. **Normalizes** scale and unit representations
3. **Grounds** claims against evidence table values
4. **Verifies** claims using three engines: direct lookup, constraint checking, and P&L recomputation
5. **Computes** a 10-dimensional signal vector summarizing verification outcomes
6. **Classifies** the answer as ACCEPT, REPAIR, or FLAG using an ML model (with hard safety gates)
7. **Repairs** REPAIR decisions deterministically and re-verifies (max 2 passes)


In [ ]:
import sys, json, os
import pandas as pd
from pathlib import Path

# Point to the project root
PROJECT_ROOT = Path(".").resolve().parent  # Adjust if running from different location
RUNS_DIR = PROJECT_ROOT / "runs"
EVAL_DIR = PROJECT_ROOT / "evaluation"

# Add backend to path for imports
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Runs directory: {RUNS_DIR}")
print(f"Evaluation directory: {EVAL_DIR}")


## 1. V6.1 ML Model Metrics

In [ ]:
# Load and display V6.1 metrics
with open(RUNS_DIR / "ml_metrics_v6_1.json") as f:
    metrics = json.load(f)

print("=== V6.1 XGBClassifier Metrics ===")
print(f"Algorithm:        {metrics['algorithm']}")
print(f"Training rows:    {metrics['training_rows']}")
print(f"Train accuracy:   {metrics['train_accuracy']:.4f}")
print(f"Val accuracy:     {metrics['val_accuracy']:.4f}")
print(f"Test accuracy:    {metrics['test_accuracy']:.4f}")
print(f"CV mean:          {metrics['cv_mean']:.4f} (+/- {metrics['cv_std']:.4f})")
print(f"CV scores:        {[round(s, 4) for s in metrics['cv_scores']]}")
print(f"FLAG recall:      {metrics['flag_recall']:.4f}")
print(f"ACCEPT recall:    {metrics['accept_recall']:.4f}")
print(f"REPAIR recall:    {metrics['repair_recall']:.4f}")
print(f"Overfit gap:      {metrics['overfit_gap']:.4f}")
print(f"Leakage detected: {metrics['leakage_detected']}")
print(f"Gate passed:      {metrics['gate_passed']}")
print(f"Label distribution: {metrics['label_distribution']}")


## 2. Training Data

In [ ]:
# Load training signals
signals_df = pd.read_csv(EVAL_DIR / "signals_v6_1_complete.csv")

print(f"Shape: {signals_df.shape}")
print(f"\nClass distribution:")
print(signals_df["label"].value_counts())
print(f"\nFirst 5 rows:")
signals_df.head()


## 3. Feature Schema

In [ ]:
with open(RUNS_DIR / "feature_schema_v6_1.json") as f:
    schema = json.load(f)

print(f"Model version: {schema['model_version']}")
print(f"Number of features: {schema['n_features']}")
print(f"\nFeature order:")
for i, feat in enumerate(schema["feature_order"], 1):
    print(f"  {i:2d}. {feat}")
print(f"\nLabel classes: {schema['label_classes']}")
print(f"Dropped from V5: {schema['dropped_from_v5']}")
print(f"Added vs V5b: {schema['added_vs_v5b']}")


## 4. Final Artifact Paths

In [ ]:
artifacts = {
    "Model": RUNS_DIR / "decision_model_v6_1.joblib",
    "Feature schema": RUNS_DIR / "feature_schema_v6_1.json",
    "Label mapping": RUNS_DIR / "label_mapping_v5.json",
    "ML metrics": RUNS_DIR / "ml_metrics_v6_1.json",
    "Model registry": RUNS_DIR / "model_registry.json",
    "Training signals": EVAL_DIR / "signals_v6_1_complete.csv",
}

print("=== Final Active Artifacts ===")
for name, path in artifacts.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = f"{size:,} bytes" if exists else "MISSING"
    print(f"  {name:20s} {path.name:40s} {status}")


## 5. Sample Verification Call

Run the full pipeline using the Python API directly (no HTTP server needed).

In [ ]:
from app.verifier.router import route_and_verify

# Apple FY2023 Income Statement
evidence = {
    "type": "table",
    "content": {
        "columns": ["", "FY2023", "FY2022"],
        "rows": [
            ["Net sales", "383,285", "394,328"],
            ["Cost of sales", "214,137", "223,546"],
            ["Gross margin", "169,148", "170,782"],
            ["Operating expenses", "54,847", "51,345"],
            ["Operating income", "114,301", "119,437"],
            ["Income taxes", "29,749", "19,300"],
            ["Net income", "96,995", "99,803"],
        ],
        "units": {}
    }
}

question = "What was Apple net sales in FY2023?"
candidate_answer = "Apple net sales were $383,285 million in FY2023."

result = route_and_verify(
    question=question,
    evidence=evidence,
    candidate_answer=candidate_answer,
    options={"tolerance": 0.01, "log_run": False}
)

print(f"Decision: {result['decision']}")
print(f"Rationale: {result['rationale']}")
print(f"Engine used: {result['engine_used']}")
print(f"Repair iterations: {result['repair_iterations']}")
print(f"Accepted after repair: {result['accepted_after_repair']}")
print(f"\nSignals:")
for k, v in result["signals"].items():
    print(f"  {k}: {v}")
print(f"\nIngestion:")
print(json.dumps(result.get("ingestion", {}), indent=2))


## 6. Evaluation Results

In [ ]:
# Load evaluation files
eval_files = {
    "Apple 20-case": "llm_evaluation_apple_20cases.json",
    "Apple adversarial": "llm_evaluation_apple_20cases_adversarial.json",
    "Hard questions 10": "llm_hard_questions_10cases.json",
    "Apple real-world": "apple_real_world_test.json",
    "Microsoft real-world": "microsoft_realworld_test.json",
    "Baseline comparison": "baseline_comparison.json",
}

print("=== Available Evaluation Files ===")
for name, filename in eval_files.items():
    path = EVAL_DIR / filename
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f"  {name}: {len(data)} cases")
        elif isinstance(data, dict):
            keys = list(data.keys())[:5]
            print(f"  {name}: {len(data)} entries (keys: {keys})")
    else:
        print(f"  {name}: FILE NOT FOUND")

# Load ablation results
print(f"\n=== Ablation Study ===")
ablation_path = EVAL_DIR / "ablation_results.csv"
if ablation_path.exists():
    ablation_df = pd.read_csv(ablation_path)
    print(ablation_df.to_string(index=False))
else:
    print("No ablation_results.csv found")


## 7. Model Evolution

In [ ]:
with open(RUNS_DIR / "model_registry.json") as f:
    registry = json.load(f)

print("=== Model Evolution ===")
print(f"Current active model: {registry['current']}\n")
for model in registry["models"]:
    v = model["version"]
    algo = model.get("algorithm", "N/A")
    status = model.get("status", "N/A")
    reason = model.get("reason", "")
    cases = model.get("training_cases", "N/A")
    taut = model.get("tautological", False)
    print(f"{v:6s} | {algo:28s} | {status:10s} | {cases} cases | tautological={taut}")
    if reason:
        print(f"       | Reason: {reason}")
    print()


## 8. Known Limitations

1. **REPAIR recall (100%) is based on only 16 cases** -- insufficient for statistical confidence
2. **TAT-QA ACCEPT cases use imputed confidence scores** -- not computed from full pipeline
3. **English P&L terminology only** -- non-English tables will have near-zero coverage
4. **Scale ambiguity** -- "383,285 million" may be parsed as billions instead of matching table units
5. **LLM-assisted ingestion** -- implemented but not empirically validated with live API
6. **No production deployment** -- no Dockerfile, no load testing, no security audit
7. **Training data is synthetic/semi-synthetic** -- cross-domain generalization is unknown

See `FINAL_SAFE_CLAIMS.md` and `FINAL_UNSAFE_CLAIMS.md` for detailed guidance on what can and cannot be claimed in the dissertation.